# Phase 9 development comparison
Run top to bottom. This notebook fits development candidates; it does not create a final production model or untouched test result. It refuses to overwrite results.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    REPO_ROOT = Path('/content/HalalifyAI')
    if not (REPO_ROOT / 'AGENTS.md').is_file():
        subprocess.run(['git', 'clone', '--branch', 'feature/model-evaluation', '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(REPO_ROOT)], check=True)
    code_paths = [Path('configs/phase9_experiment_protocol_v1.json'), Path('src/model_training/phase9_experiment.py')]
    missing_code = [p for p in code_paths if not (REPO_ROOT / p).is_file()]
    if missing_code:
        from google.colab import files
        from io import BytesIO
        print('Upload phase9_code.zip containing:', *map(str, missing_code))
        archive = files.upload().get('phase9_code.zip')
        if archive is None:
            raise FileNotFoundError('Upload phase9_code.zip')
        with zipfile.ZipFile(BytesIO(archive)) as bundle:
            expected = {p.as_posix() for p in missing_code}
            if set(bundle.namelist()) != expected or any('..' in Path(n).parts or Path(n).is_absolute() for n in bundle.namelist()):
                raise ValueError(f'Code ZIP must contain exactly: {sorted(expected)}')
            bundle.extractall(REPO_ROOT)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scipy', 'scikit-learn', 'exchange_calendars'], check=True)
else:
    start = Path.cwd().resolve()
    REPO_ROOT = next((p for p in [start, *start.parents] if (p / 'AGENTS.md').is_file() and (p / 'src').is_dir() and (p / 'configs').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Open Jupyter inside the HalalifyAI repository.')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Environment:', 'Colab' if IN_COLAB else 'Local', 'Repository:', REPO_ROOT)


In [ ]:
import json, zipfile
from src.model_training.phase9_experiment import verify_inputs
CONTRACT = Path('configs/phase9_experiment_protocol_v1.json')
spec = json.loads(CONTRACT.read_text(encoding='utf-8'))
required = [Path(spec['source_dataset']), Path(spec['fold_source']), Path('reports/model_evaluation/phase7_candidate_tuning_v1/experiment_contract.json'), Path('reports/model_evaluation/phase7_candidate_tuning_v1/return/logistic_regression/candidate_report.json'), Path('reports/model_evaluation/phase7_candidate_tuning_v1/risk/hist_gradient_boosting/candidate_report.json')]
print('Required inputs and destinations:')
for path in required:
    print(path, 'present' if path.is_file() else 'MISSING')
if IN_COLAB and any(not path.is_file() for path in required):
    from google.colab import files
    print('Upload phase9_tuning_inputs.zip with these files at the paths printed above.')
    uploaded = files.upload()
    archive = uploaded.get('phase9_tuning_inputs.zip')
    if archive is None:
        raise FileNotFoundError('Upload phase9_tuning_inputs.zip')
    from io import BytesIO
    with zipfile.ZipFile(BytesIO(archive)) as bundle:
        expected = {p.as_posix() for p in required if not p.is_file()}
        if set(bundle.namelist()) != expected or any('..' in Path(n).parts or Path(n).is_absolute() for n in bundle.namelist()):
            raise ValueError(f'ZIP must contain exactly missing paths: {sorted(expected)}')
        bundle.extractall(REPO_ROOT)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Restore required inputs at: ' + ', '.join(missing))
OUTPUT = Path('reports/model_evaluation/phase9_tuning_v1')
if OUTPUT.exists():
    raise FileExistsError(f'Existing result directory; preserve it: {OUTPUT}')
contract, frame, protocol = verify_inputs(CONTRACT)
print('Preflight passed:', len(frame), 'rows;', len(protocol['folds']), 'folds')


In [ ]:
from src.model_training.phase9_experiment import run_comparison
OUTPUT = Path('reports/model_evaluation/phase9_tuning_v1')
result = run_comparison(CONTRACT, OUTPUT)
print(json.dumps(result, indent=2))
print('Development comparison only. Inspect every fold before selecting a model.')


In [ ]:
import pandas as pd
results = json.loads((OUTPUT / 'candidate_results.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'task': r['task'], 'fold': r['fold'], 'factor': r['factor'], 'selected': r['variant'], **r['metrics']} for r in results]))
print('Files:', *sorted(str(p) for p in OUTPUT.iterdir()), sep='\n')
if IN_COLAB:
    from google.colab import files
    import shutil
    archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name)
    files.download(archive)
